#  IshVenom — Gemma 4 E2B LoRA Fine-Tuning

Fine-tune **Gemma 4 E2B** with LoRA using **Unsloth** for fast,
memory-efficient training. Produces a **GGUF Q4_K_M** file that
runs on-device via `llama.cpp` / `llama.rn`.

### Kaggle Settings
| Setting | Value |
|---------|-------|
| Accelerator | GPU T4 × 2 |
| Internet | ON |
| Attached Dataset | `kwakyeishmael/ishvenom-corpus` |
| Secret | `HF_TOKEN` (HuggingFace token with Gemma access) |

### Outputs (saved to `/kaggle/working/`)
- `ishvenom-gemma4-e2b-q4km/` — GGUF Q4_K_M directory (1.3–1.6 GB)

**Runtime:** ~4–6 hours on T4 × 2

>  You must accept the Gemma 4 license at
> https://huggingface.co/google/gemma-4-e2b-it before running this.

## 1 · Load HuggingFace Token
The HF token is stored as a Kaggle Secret named `HF_TOKEN`.
This is needed to download the gated Gemma 4 model from HuggingFace.

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print(" HF_TOKEN loaded from Kaggle Secrets")
except Exception:
    # Fallback: set manually if not on Kaggle
    if "HF_TOKEN" not in os.environ:
        print(" HF_TOKEN not found.")
        print("   On Kaggle: Add it via Notebook Settings → Secrets → HF_TOKEN")
        print("   Locally:   export HF_TOKEN=<your-token>  (never hardcode in notebook)")
    else:
        print(" HF_TOKEN found in environment")

## 2 · Install Unsloth
Unsloth provides 2x faster LoRA training and built-in GGUF export.
The Kaggle × Unsloth prize requires using Unsloth — we comply.

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "unsloth",
    "trl>=0.11.0",
])
print(" Unsloth + TRL installed")

## 3 · Configuration
All hyperparameters and paths in one place.
The corpus dataset is attached at `/kaggle/input/ishvenom-corpus/`.

In [ ]:
from pathlib import Path

# ── Paths ──
BASE_MODEL   = "google/gemma-4-e2b-it"
# Updated path to match Kaggle's actual mount location
CORPUS_PATH  = Path("/kaggle/input/datasets/kwakyeishmael/ishvenom-corpus/data/processed/corpus/firstaid_sft.jsonl")
OUT_DIR      = Path("/kaggle/working")

# ── LoRA Config ──
LORA_RANK    = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# ── Training Config ──
EPOCHS       = 3
BATCH_SIZE   = 1      # Reduced from 4/8 to prevent OOM
GRAD_ACCUM   = 16     # Increased so effective batch size is still 16
LR           = 2e-4
WARMUP_RATIO = 0.03
MAX_SEQ_LEN  = 1024
SEED         = 42

# ── Output ──
GGUF_OUT     = str(OUT_DIR / "ishvenom-gemma4-e2b-q4km")

print(f"Base model   : {BASE_MODEL}")
print(f"Corpus       : {CORPUS_PATH}")
print(f"LoRA rank    : {LORA_RANK}, alpha: {LORA_ALPHA}")
print(f"Epochs       : {EPOCHS}, effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Corpus exists: {CORPUS_PATH.exists()}")

## 4 · Load Training Corpus
Read the JSONL file produced by the IshVenom data pipeline.
Each row has `instruction` (the user question) and `output` (the
first-aid response), plus a `language` field.

In [ ]:
import json

raw_rows: list[dict] = []
with CORPUS_PATH.open(encoding="utf-8") as f:
    for line in f:
        raw_rows.append(json.loads(line.strip()))

print(f" Loaded {len(raw_rows):,} training examples")

# Count by language
by_lang: dict[str, int] = {}
for row in raw_rows:
    lang = row.get("language", "?")
    by_lang[lang] = by_lang.get(lang, 0) + 1

print(f"\nLanguage distribution:")
for lang, count in sorted(by_lang.items()):
    print(f"  {lang}: {count:,} examples")

## 5 · Format Examples with Gemma Chat Template
Convert each instruction/output pair into Gemma's chat format:
```
<start_of_turn>user
{instruction}<end_of_turn>
<start_of_turn>model
{output}<end_of_turn>
```

In [ ]:
def format_example(row: dict) -> str:
    """Format a single row into Gemma 4 chat template."""
    instruction = row["instruction"].strip()
    output      = row["output"].strip()
    return (
        f"<start_of_turn>user\n{instruction}<end_of_turn>\n"
        f"<start_of_turn>model\n{output}<end_of_turn>"
    )


formatted = [{"text": format_example(r)} for r in raw_rows]

print(f" {len(formatted):,} examples formatted")
print("\n── Sample (first 300 chars) ──")
print(formatted[0]["text"][:300])

## 6 · Load Base Model with Unsloth
Load `google/gemma-4-e2b-it` using Unsloth's `FastLanguageModel`.
Uses 4-bit quantization to fit on T4 (16 GB VRAM per GPU).

> This step downloads ~5 GB from HuggingFace. Make sure Internet is ON.

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,        # auto-detect: bf16 on A100, fp16 on T4
    load_in_4bit=True, # 4-bit quantization for T4 (16 GB VRAM)
)

print(f" Base model loaded: {BASE_MODEL}")
print(f"   Dtype: {next(model.parameters()).dtype}")

## 7 · Add LoRA Adapters
Attach low-rank adapters to all linear projection layers in Gemma.
Only the LoRA weights are trainable — the base model stays frozen.

| Param | Value |
|-------|-------|
| Rank | 16 |
| Alpha | 32 |
| Target modules | q, k, v, o, gate, up, down projections |
| Trainable params | ~0.5% of total |

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's memory-efficient checkpointing
    random_state=SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f" LoRA adapters attached")
print(f"   Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

## 8 · Train with SFTTrainer
Use HuggingFace's `SFTTrainer` (Supervised Fine-Tuning Trainer) to train
the LoRA adapters. Logs every 10 steps. Saves checkpoint each epoch.

**Expected runtime: ~4-6 hours on T4 × 2**

In [ ]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

dataset = Dataset.from_list(formatted)

use_bf16 = torch.cuda.is_bf16_supported()
print(f"Mixed precision: {'bf16' if use_bf16 else 'fp16'}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=WARMUP_RATIO,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        fp16=not use_bf16,
        bf16=use_bf16,
        logging_steps=10,
        output_dir=str(OUT_DIR / "trainer_output"),
        save_strategy="epoch",
        save_total_limit=1,
        seed=SEED,
        report_to="none",  # no wandb on Kaggle
    ),
)

print(f"\n Starting training:")
print(f"   Epochs          : {EPOCHS}")
print(f"   Effective batch : {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Learning rate   : {LR}")

trainer_stats = trainer.train()

print(f"\n Training complete!")
print(f"   Runtime     : {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"   Samples/sec : {trainer_stats.metrics['train_samples_per_second']:.2f}")
print(f"   Final loss  : {trainer_stats.metrics['train_loss']:.4f}")

## 9 · Export to GGUF Q4_K_M
Unsloth merges the LoRA weights into the base model and quantizes to
**GGUF Q4_K_M** format in one call. No need for a separate `llama.cpp` build.

This is the file that `llama.rn` loads on the phone.
Target file size: **< 1.6 GB**.

In [ ]:
import os

# IMPORTANT: Replace with your actual Hugging Face username
HF_USERNAME = "CalyxIsh" 
REPO_NAME = f"{HF_USERNAME}/ishvenom-gemma-e2b-merged"

HF_WRITE_TOKEN = os.environ.get("HF_TOKEN", "")  # loaded from Kaggle Secret in cell 1

print(f"Pushing merged 16-bit model to Hugging Face -> {REPO_NAME}")
print("This bypasses the GGUF disk limits on Kaggle...")

model.push_to_hub_merged(
    REPO_NAME,
    tokenizer,
    save_method="merged_16bit",
    token=HF_WRITE_TOKEN,
)

print("Model export complete!")
print(f"Your model is now live at: https://huggingface.co/{REPO_NAME}")

## 10 · Final Summary
Report the output file sizes and give download instructions.

In [ ]:
try:
    repo_link = f"https://huggingface.co/{REPO_NAME}/tree/main"
except NameError:
    repo_link = "your Hugging Face repository"

print("=" * 60)
print(" ISHVENOM MODEL DEPLOYMENT SUMMARY")
print("=" * 60)
print("\nYour fine-tuned model has been successfully pushed to the cloud.")
print("Because we bypassed Kaggle's disk limits, local file search is skipped.")
print(f"\nModel Location: {repo_link}")
print("\nNext Steps for Mobile Integration:")
print("   1. Download the generated Q4_K_M .gguf file from the Hugging Face link above.")
print("   2. Rename the downloaded file to: gemma-4-e2b-q4km.gguf")
print("   3. Move it to your project folder:")
print("      -> apps/mobile/assets/models/gemma-4-e2b-q4km.gguf")
print("\nPipeline complete.")